# Processamento Digital de Sinais - 2ª Avaliação

## Questão 2 (3,0 pontos) - Espectro da TFTD de um pulso retangular

Este notebook obtém o espectro da **Transformada de Fourier em Tempo Discreto (TFTD)** do sinal
proposto no enunciado — um pulso retangular de amplitude unitária —, com o código **generalizado
para qualquer largura**.

O sinal do enunciado é

$$x[n] = \begin{cases} 1, & -4 \le n \le 4\\ 0, & \text{caso contrário}\end{cases}$$

que corresponde ao caso particular $M = 4$ da família

$$x_M[n] = \begin{cases} 1, & |n| \le M\\ 0, & \text{caso contrário}\end{cases}$$

com $L = 2M + 1$ amostras não nulas. O parâmetro `M` controla a largura e pode ser alterado
livremente.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Todos os graficos sao salvos nesta pasta (criada se nao existir)
IMAGES_DIR = 'images'
os.makedirs(IMAGES_DIR, exist_ok=True)

plt.rcParams['figure.dpi'] = 110

## 1. Geração do sinal (largura parametrizável)

A função abaixo constrói $x_M[n]$ para qualquer largura `M`, devolvendo tanto o vetor de índices
$n$ quanto as amostras. É essa parametrização que atende à exigência de *"generalizar o código
para que seja possível alterar a largura do sinal"*.

In [ ]:
def pulso_retangular(M, margem=4):
    """Gera x[n] = 1 para |n| <= M e 0 fora, com algumas amostras nulas nas bordas.

    M      : semilargura do pulso (o pulso tem L = 2M+1 amostras unitarias)
    margem : quantas amostras nulas mostrar de cada lado (apenas para visualizacao)
    Retorna (n, x).
    """
    if M < 0:
        raise ValueError("M deve ser >= 0.")
    n = np.arange(-M - margem, M + margem + 1)
    x = np.where(np.abs(n) <= M, 1.0, 0.0)
    return n, x


# Caso do enunciado
M = 4
n, x = pulso_retangular(M)
L = 2*M + 1

print(f"M = {M}  ->  L = 2M+1 = {L} amostras unitarias")
print("n    =", n)
print("x[n] =", x.astype(int))

## 2. TFTD pela definição

Por definição, a TFTD de uma sequência aperiódica é

$$X(e^{j\omega}) = \sum_{n=-\infty}^{\infty} x[n]\,e^{-j\omega n}.$$

Como $x_M[n]$ tem duração finita, o somatório se reduz a $-M \le n \le M$:

$$X_M(e^{j\omega}) = \sum_{n=-M}^{M} e^{-j\omega n}.$$

A função a seguir avalia essa soma diretamente, sem usar nenhuma rotina pronta de transformada.

In [ ]:
def tftd_definicao(x, n, w):
    """Avalia X(e^{jw}) = sum_n x[n] e^{-j w n} diretamente pela definicao.

    x, n : amostras do sinal e seus indices
    w    : vetor de frequencias (rad/amostra)
    """
    x = np.asarray(x, dtype=float)
    n = np.asarray(n)
    # matriz de expoentes: uma linha por frequencia, uma coluna por amostra
    E = np.exp(-1j * np.outer(w, n))
    return E @ x


w = np.linspace(-np.pi, np.pi, 2001)
X_def = tftd_definicao(x, n, w)

print("X(e^{j0}) =", np.real_if_close(tftd_definicao(x, n, np.array([0.0])))[0],
      f" (esperado L = {L})")

## 3. Forma fechada — núcleo de Dirichlet

A soma acima é uma progressão geométrica e admite forma fechada. Escrevendo $r = e^{-j\omega}$:

$$\sum_{n=-M}^{M} r^{n} = \frac{r^{-M-\frac12} - r^{M+\frac12}}{r^{-\frac12} - r^{\frac12}}
= \frac{e^{j\omega\left(M+\frac12\right)} - e^{-j\omega\left(M+\frac12\right)}}{e^{j\omega/2} - e^{-j\omega/2}}
= \frac{2j\,\mathrm{sen}\!\left[\omega\left(M+\frac12\right)\right]}{2j\,\mathrm{sen}(\omega/2)}$$

ou seja,

$$\boxed{\;X_M(e^{j\omega}) = \frac{\mathrm{sen}\!\left(\dfrac{\omega L}{2}\right)}{\mathrm{sen}\!\left(\dfrac{\omega}{2}\right)},\qquad L = 2M+1\;}$$

conhecida como **núcleo de Dirichlet** (ou *sinc periódica*). Observações importantes:

- $X_M(e^{j\omega})$ é **puramente real**, consequência da simetria par de $x_M[n]$;
- em $\omega = 0$ o limite vale $X_M(e^{j0}) = L$ (soma de todas as amostras);
- os cruzamentos por zero ocorrem em $\omega = \dfrac{2\pi k}{L}$, com $k$ inteiro não múltiplo de $L$;
- a **largura do lóbulo principal** é $\dfrac{4\pi}{L}$ — inversamente proporcional à largura do pulso.

In [ ]:
def tftd_dirichlet(M, w):
    """Forma fechada da TFTD do pulso retangular: sen(wL/2)/sen(w/2), L = 2M+1."""
    L = 2*M + 1
    den = np.sin(w/2)
    # nos multiplos de 2*pi o quociente e indeterminado; o limite vale +-L
    quase_zero = np.abs(den) < 1e-12
    X = np.empty_like(w, dtype=float)
    X[~quase_zero] = np.sin(w[~quase_zero]*L/2) / den[~quase_zero]
    # limite: sen(wL/2)/sen(w/2) -> L * cos(...)/cos(...) = +-L
    X[quase_zero] = L * np.cos(w[quase_zero]*L/2) / np.cos(w[quase_zero]/2)
    return X


X_dir = tftd_dirichlet(M, w)

erro = np.max(np.abs(X_def - X_dir))
print(f"Erro maximo entre a definicao e a forma fechada : {erro:.2e}")
print(f"Maxima parte imaginaria da definicao            : {np.max(np.abs(X_def.imag)):.2e}"
      "   (confirma que X e real)")

## 4. Cálculo via FFT

O enunciado permite usar rotinas prontas de Transformada Rápida de Fourier. A FFT calcula a
**DFT**, que corresponde a **amostrar a TFTD** em $N$ pontos igualmente espaçados:

$$X[k] = X\!\left(e^{j\omega}\right)\Big|_{\omega = \frac{2\pi k}{N}}, \qquad k = 0,1,\dots,N-1.$$

Dois cuidados são necessários:

1. **Indexação circular.** A DFT assume que a sequência começa em $n = 0$. Como $x_M[n]$ é definido
   em $-M \le n \le M$, as amostras de índice negativo são colocadas no **final** do vetor
   (equivalente à periodicidade da DFT). Isso preserva a fase corretamente e o resultado sai real.
2. **Zero-padding.** Aumentar $N$ não cria informação nova: apenas amostra a TFTD mais densamente,
   fazendo a curva contínua aparecer com mais detalhe.

In [ ]:
def tftd_via_fft(M, N=1024):
    """Amostra a TFTD do pulso via FFT, com zero-padding em N pontos.

    Retorna (w, X) ordenados em -pi <= w < pi.
    """
    if N < 2*M + 1:
        raise ValueError("N deve ser >= L = 2M+1 para nao truncar o sinal.")
    xz = np.zeros(N)
    xz[:M+1] = 1.0        # amostras n = 0, 1, ..., M
    if M > 0:
        xz[N-M:] = 1.0    # amostras n = -M, ..., -1 (envolvimento circular)

    X = np.fft.fft(xz)
    w = 2*np.pi*np.arange(N)/N
    w = np.where(w >= np.pi, w - 2*np.pi, w)   # leva para -pi <= w < pi

    ordem = np.argsort(w)
    return w[ordem], X[ordem]


w_fft, X_fft = tftd_via_fft(M, N=1024)

# compara as amostras da FFT com a forma fechada avaliada nas mesmas frequencias
erro_fft = np.max(np.abs(X_fft - tftd_dirichlet(M, w_fft)))
print(f"Erro maximo entre a FFT e a forma fechada : {erro_fft:.2e}")
print(f"Maxima parte imaginaria da FFT            : {np.max(np.abs(X_fft.imag)):.2e}")

## 5. Gráficos

### 5.1 O sinal no tempo

In [ ]:
plt.figure(figsize=(9, 3.2))
plt.stem(n, x, basefmt=' ')
plt.xlabel('$n$')
plt.ylabel('$x[n]$')
plt.title(f'Pulso retangular $x[n]$ — $M = {M}$ ($L = {L}$ amostras)')
plt.ylim(-0.15, 1.25)
plt.xticks(np.arange(n[0], n[-1]+1, 2))
plt.grid(True, alpha=0.4)
plt.savefig(os.path.join(IMAGES_DIR, 'sinal_tempo.png'), dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Espectro: magnitude e fase

Como $X_M(e^{j\omega})$ é real, é informativo mostrar **três** curvas:

- a função $X_M(e^{j\omega})$ com sinal (onde se enxergam os lóbulos negativos);
- a **magnitude** $|X_M(e^{j\omega})|$;
- a **fase** $\angle X_M(e^{j\omega})$, que só assume os valores $0$ (onde $X>0$) e $\pi$ (onde $X<0$).

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 9))

ticks = [-np.pi, -np.pi/2, 0, np.pi/2, np.pi]
rotulos = [r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$']

# (a) parte real com sinal + amostras da FFT
ax1.plot(w, X_dir, 'C0', label='TFTD (forma fechada)')
ax1.plot(w_fft[::8], np.real(X_fft[::8]), 'C3.', ms=4, label='amostras da FFT')
ax1.axhline(0, color='k', lw=0.8)
ax1.set_ylabel(r'$X(e^{j\omega})$')
ax1.set_title(f'TFTD do pulso retangular — $M = {M}$, $L = {L}$')
ax1.legend(loc='upper right', fontsize=9)

# (b) magnitude
ax2.plot(w, np.abs(X_dir), 'C0')
ax2.axvline(2*np.pi/L, color='gray', ls=':', lw=1)
ax2.axvline(-2*np.pi/L, color='gray', ls=':', lw=1)
ax2.set_ylabel(r'$|X(e^{j\omega})|$')
ax2.set_title(r'Magnitude — linhas pontilhadas: primeiro zero em $\omega = \pm 2\pi/L$')

# (c) fase
fase = np.angle(X_dir.astype(complex))
ax3.plot(w, fase, 'C1')
ax3.set_ylabel(r'$\angle X(e^{j\omega})$ (rad)')
ax3.set_xlabel(r'$\omega$ (rad/amostra)')
ax3.set_title(r'Fase — alterna entre $0$ e $\pi$ a cada troca de sinal')
ax3.set_yticks([-np.pi, 0, np.pi])
ax3.set_yticklabels([r'$-\pi$', '0', r'$\pi$'])

for a in (ax1, ax2, ax3):
    a.set_xlim(-np.pi, np.pi)
    a.set_xticks(ticks)
    a.set_xticklabels(rotulos)
    a.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'espectro_M4.png'), dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Efeito da largura do sinal

Aqui se explora a generalização pedida. Variando $M$, observa-se a **relação inversa** entre a
duração no tempo e a largura em frequência: quanto mais **largo** o pulso, mais **estreito** o
lóbulo principal do espectro. O pico central vale sempre $L = 2M+1$, e a largura do lóbulo
principal é $4\pi/L$.

In [ ]:
valores_M = [2, 4, 8, 16]

fig, (axA, axB) = plt.subplots(2, 1, figsize=(10, 7))

for i, Mi in enumerate(valores_M):
    ni, xi = pulso_retangular(Mi, margem=2)
    base = i*1.5
    axA.stem(ni, xi + base, bottom=base, basefmt=' ',
             linefmt=f'C{i}-', markerfmt=f'C{i}o')
    axA.axhline(base, color='0.85', lw=0.8, zorder=0)
    axA.text(-19.5, base + 0.35, f'$M = {Mi}$', color=f'C{i}',
             fontsize=10, va='center')

axA.set_xlabel('$n$')
axA.set_title('Pulsos de larguras diferentes (deslocados verticalmente)')
axA.set_yticks([])
axA.set_xlim(-20, 20)
axA.grid(True, alpha=0.3, axis='x')

for i, Mi in enumerate(valores_M):
    Li = 2*Mi + 1
    axB.plot(w, np.abs(tftd_dirichlet(Mi, w)), f'C{i}',
             label=f'$M = {Mi}$  ($L = {Li}$, lóbulo $= {4/Li:.2f}\\pi$)')

axB.set_xlabel(r'$\omega$ (rad/amostra)')
axB.set_ylabel(r'$|X(e^{j\omega})|$')
axB.set_title('Magnitude do espectro: pulso mais largo no tempo, lóbulo mais estreito em frequência')
axB.set_xlim(-np.pi, np.pi)
axB.set_xticks(ticks)
axB.set_xticklabels(rotulos)
axB.legend(loc='upper right', fontsize=9)
axB.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'efeito_largura.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"{'M':>3} {'L=2M+1':>7} {'X(0)':>6} {'1o zero':>12} {'lobulo principal':>18}")
for Mi in valores_M:
    Li = 2*Mi + 1
    print(f"{Mi:>3} {Li:>7} {Li:>6} {2/Li:>10.4f}pi {4/Li:>16.4f}pi")

### 5.4 A DFT como amostragem da TFTD

O gráfico abaixo torna explícita a relação usada no item (iv) da pesquisa: a **FFT calcula a DFT**,
que nada mais é do que a TFTD **amostrada** em $N$ pontos. Com $N$ pequeno as amostras são
esparsas; aumentando $N$ (zero-padding), elas se adensam e reconstroem visualmente a curva
contínua — sem acrescentar informação nova ao sinal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)

for ax, N in zip(axes, [16, 32, 128]):
    wN, XN = tftd_via_fft(M, N=N)
    ax.plot(w, np.abs(X_dir), 'C0-', lw=1.2, label='TFTD (contínua)')
    ax.stem(wN, np.abs(XN), linefmt='C3-', markerfmt='C3o', basefmt=' ',
            label=f'DFT, $N = {N}$')
    ax.set_xlim(-np.pi, np.pi)
    ax.set_xticks(ticks)
    ax.set_xticklabels(rotulos)
    ax.set_xlabel(r'$\omega$')
    ax.set_title(f'$N = {N}$ pontos')
    ax.grid(True, alpha=0.4)

axes[0].set_ylabel(r'$|X(e^{j\omega})|$')
axes[0].legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, 'dft_amostra_tftd.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Verificação final

Confronto dos três caminhos de cálculo — definição, forma fechada e FFT — para várias larguras,
garantindo que a implementação generalizada está correta.

In [ ]:
print(f"{'M':>3} {'L':>4} {'|def - fechada|':>18} {'|FFT - fechada|':>18} {'Im(X) max':>12}")
for Mi in [0, 1, 2, 4, 8, 16, 32]:
    ni, xi = pulso_retangular(Mi, margem=1)
    Xd = tftd_definicao(xi, ni, w)
    Xa = tftd_dirichlet(Mi, w)
    wf, Xf = tftd_via_fft(Mi, N=1024)
    e1 = np.max(np.abs(Xd - Xa))
    e2 = np.max(np.abs(Xf - tftd_dirichlet(Mi, wf)))
    im = np.max(np.abs(Xd.imag))
    print(f"{Mi:>3} {2*Mi+1:>4} {e1:>18.3e} {e2:>18.3e} {im:>12.3e}")

# Teorema de Parseval: energia no tempo = energia em frequencia
wp = np.linspace(-np.pi, np.pi, 200001)
energia_tempo = np.sum(x**2)
integra = getattr(np, 'trapezoid', None) or np.trapz
energia_freq = integra(np.abs(tftd_dirichlet(M, wp))**2, wp)/(2*np.pi)
print(f"\nParseval:  soma|x[n]|^2 = {energia_tempo:.6f}   "
      f"(1/2pi)int|X|^2 dw = {energia_freq:.6f}   "
      f"erro = {abs(energia_tempo-energia_freq):.2e}")